# ViT-ESI with 185 electrodes on Kaggle -- generate data (SEREEGA) -> train -> test -> package, fully automatic

**Before running** (Settings panel, right side): **Accelerator -> GPU T4 x2**, **Internet -> On**. No dataset has to be attached.
Then **Run All**. Everything below happens automatically:

| Step | What happens | Files written (in `results_185/`) |
|------|--------------|-----------------------------------|
| 1 | clone the repo, install packages, record the environment | `run_manifest.json`, `logs/` |
| 2 | generate **6000 training + 2000 test** samples with SEREEGA (185-electrode leadfield, the previous team's parameters) and verify them | `logs/data_manifest.json`, `logs/generate_*.log` |
| 3 | train ViT-ESI on **both T4 GPUs**, 100 epochs, logging every step/epoch; the model is saved after every epoch (the same file is overwritten) | `training/` (`training_epochs.csv`, `training_steps.csv`, `last_model.pt`, `last.ckpt`, `best_val_model.pt`, ...) |
| 4 | test on the generated 2000-sample test set with the team's `eval.py` (LE, AUC, nMSE, PSNR, time error) | `test/` (per-sample metrics with ids, summary, example predictions) |
| 5 | figures + summary report | `figures/`, `SUMMARY.md` |
| 6 | zip everything (never the data) | **`/kaggle/working/results_185.zip`** |

Real-data validation is done afterwards on the local machine with the downloaded zip. The generated data go to `/kaggle/temp` (not part of the notebook output).
Set `QUICK_TEST = True` in the next cell for a ~10-minute dry run of the whole pipeline (60 train / 20 test samples, 2 epochs).

In [ ]:
# ============================ settings (the defaults reproduce the previous team's setup with 185 electrodes) ============================
REPO_URL = "https://github.com/fernandonpa/185_head-training.git"
BRANCH = "main"

QUICK_TEST = False          # True: tiny dry run of the full pipeline (checks everything works on this machine)

N_TRAIN, N_TEST = (60, 20) if QUICK_TEST else (6000, 2000)      # samples generated with SEREEGA
SEED_TRAIN, SEED_TEST = 0, 1                                      # different seeds -> independent random draws
N_EPOCHS = 2 if QUICK_TEST else 100
BATCH_PER_GPU = 4 if QUICK_TEST else 32     # the team's Kaggle notebook: 32 per GPU (thesis: effective batch 8). Lower it if you want to stay closer to the thesis.
NUM_WORKERS = 2                             # data-loading workers per GPU process
PRECISION = "16-mixed"                      # the team's Kaggle notebook value ("32-true" = full precision, slower)
EEG_SNR = 5                                 # dB, white noise added during training/testing (team: 5)
PER_VALID = 0.2                             # validation share of the training set (team: 0.2)
EVAL_WEIGHTS = "best_val"                   # model used for the test: "best_val" (lowest validation loss) or "last" (final epoch)
RESUME = False                              # True: continue from results_185/training/last.ckpt if it exists (same session only)
# ===========================================================================================================================================

In [ ]:
import os, sys, json, glob, time, shutil, subprocess, platform
from pathlib import Path

WORK = Path("/kaggle/working")
TEMP = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else WORK        # /kaggle/temp is not saved with the notebook output
REPO_DIR = WORK / "185_head-training"
DATA_ROOT = TEMP / "data_185"
RESULTS = WORK / "results_185"
LOGS, TRAIN_REC, TEST_REC, FIGS = RESULTS / "logs", RESULTS / "training", RESULTS / "test", RESULTS / "figures"
for d in (RESULTS, LOGS, TRAIN_REC, TEST_REC, FIGS):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_NAME, TEST_NAME = f"train_{N_TRAIN}", f"test_{N_TEST}"
T_START = time.time()
STEP_TIMES = {}

def run(cmd, log_name, env=None, cwd=None):
    # run a command, show its output live and keep a copy in results_185/logs/<log_name>; stop the notebook if it fails
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    e = dict(os.environ, PYTHONUNBUFFERED="1", **(env or {}))
    t0 = time.time()
    with open(LOGS / log_name, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, cwd=cwd, env=e, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); log.write(line)
        p.wait()
    print(f"[{log_name}] exit code {p.returncode}, {time.time() - t0:.0f}s", flush=True)
    if p.returncode != 0:
        raise RuntimeError(f"{log_name} failed (exit {p.returncode}); see {LOGS / log_name}")

def snapshot_gpu(tag):
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        (LOGS / f"nvidia_smi_{tag}.txt").write_text(r.stdout or r.stderr)
    except FileNotFoundError:
        pass

## 1. Check the GPUs, get the code, record the environment

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", n_gpus)
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
if n_gpus == 0:
    raise RuntimeError("No GPU: set Accelerator -> GPU T4 x2 in the Settings panel and restart.")
if n_gpus < 2:
    print("WARNING: only one GPU is visible; training still works but is slower.")

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print("repo:", REPO_DIR, "| commit", COMMIT)
print("generated data go to:", DATA_ROOT, "(/kaggle/temp is not saved with the notebook output)" if str(DATA_ROOT).startswith("/kaggle/temp") else "(WARNING: /kaggle/temp not found, data will count towards the notebook output)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytorch-lightning", "mne", "colorednoise", "scikit-image"], check=True)

import pytorch_lightning as pl, numpy as np, scipy
LEADFIELD = REPO_DIR / "anatomy" / "leadfield_185_20k.mat"
assert LEADFIELD.is_file(), f"missing {LEADFIELD}"
manifest = dict(
    created=time.strftime("%Y-%m-%d %H:%M:%S"), git_commit=COMMIT, repo=REPO_URL, branch=BRANCH,
    settings=dict(QUICK_TEST=QUICK_TEST, N_TRAIN=N_TRAIN, N_TEST=N_TEST, SEED_TRAIN=SEED_TRAIN, SEED_TEST=SEED_TEST, N_EPOCHS=N_EPOCHS, BATCH_PER_GPU=BATCH_PER_GPU,
                  GLOBAL_BATCH=BATCH_PER_GPU * max(1, n_gpus), NUM_WORKERS=NUM_WORKERS, PRECISION=PRECISION, EEG_SNR=EEG_SNR, PER_VALID=PER_VALID, EVAL_WEIGHTS=EVAL_WEIGHTS),
    environment=dict(python=sys.version.split()[0], torch=torch.__version__, pytorch_lightning=pl.__version__, numpy=np.__version__, scipy=scipy.__version__,
                     platform=platform.platform(), cpu_count=os.cpu_count(), gpus=[torch.cuda.get_device_name(i) for i in range(n_gpus)]),
    paths=dict(data_root=str(DATA_ROOT), results=str(RESULTS)))
(RESULTS / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
snapshot_gpu("before_training")
print(json.dumps(manifest["settings"], indent=1))

## 2. Generate the SEREEGA data (185 electrodes) and verify it

In [ ]:
t0 = time.time()
run([sys.executable, REPO_DIR / "kaggle_185" / "generate_data_185.py", "--root", DATA_ROOT, "--logs_dir", LOGS,
     "--n_train", N_TRAIN, "--n_test", N_TEST, "--train_name", TRAIN_NAME, "--test_name", TEST_NAME,
     "--seed_train", SEED_TRAIN, "--seed_test", SEED_TEST], "data_generation.log", cwd=REPO_DIR)
STEP_TIMES["data_generation_s"] = time.time() - t0
dm = json.load(open(DATA_ROOT / "data_manifest.json"))
for k, v in dm["splits"].items():
    print(k, "| samples", v["n_eeg_files"], "| patches", v["n_patch_hist"], "| orders", v["order_hist"], "| problems", v["problems"] or "none")

In [ ]:
# look at one generated sample and keep the picture
import numpy as np, matplotlib.pyplot as plt
from scipy.io import loadmat
simu = DATA_ROOT / "simulation" / "fsaverage" / "constrained" / "standard_1020" / "fsav_994" / "simu"
eeg = loadmat(simu / TRAIN_NAME / "eeg" / "infdb" / "1_eeg.mat", simplify_cells=True)["eeg_data"]["EEG"]
src = np.atleast_2d(loadmat(simu / TRAIN_NAME / "sources" / "Jact" / "1_src_act.mat", simplify_cells=True)["Jact"]["Jact"])
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].plot(eeg.T, lw=0.3); ax[0].set_title(f"sample 1: EEG {eeg.shape} (185 electrodes, 500 samples at 1000 Hz, before noise)")
ax[1].plot(src.T, lw=0.5); ax[1].set_title(f"active source waveforms ({src.shape[0]} regions)")
fig.tight_layout(); fig.savefig(FIGS / "fig_sample_data.png", dpi=120); plt.show()

## 3. Train ViT-ESI on both GPUs (records every step and epoch; one model file updated after every epoch)

In [ ]:
t0 = time.time()
train_simu_folder = simu / TRAIN_NAME
train_cmd = [sys.executable, REPO_DIR / "kaggle_185" / "train_185.py", TRAIN_NAME,
    "-results_path", RESULTS / "main_train_output",
    "-simu_type", "sereega", "-source_space", "fsav_994", "-electrode_montage", "standard_1020", "-orientation", "constrained",
    "-model", "VIT", "-to_load", N_TRAIN, "-per_valid", PER_VALID, "-n_times", 500, "-eeg_snr", EEG_SNR,
    "-loss", "cosine", "-scaler", "linear", "-batch_size", BATCH_PER_GPU, "-num_workers", NUM_WORKERS, "-precision", PRECISION,
    "-leadfield_mat", LEADFIELD, "-vit_embed_dim", 256, "-vit_depth", 6, "-vit_heads", 8, "-vit_mlp_dim", 512, "-vit_dropout", 0.1,
    "-simu_folder", train_simu_folder, "-n_epochs", N_EPOCHS]
if RESUME and (TRAIN_REC / "last.ckpt").exists():
    train_cmd += ["-resume", TRAIN_REC / "last.ckpt"]
    print("resuming from", TRAIN_REC / "last.ckpt")
run(train_cmd, "training.log", env={"ESI185_RECORDS_DIR": str(TRAIN_REC)}, cwd=REPO_DIR / "model_training")
STEP_TIMES["training_s"] = time.time() - t0
snapshot_gpu("after_training")

In [ ]:
import pandas as pd
ep = pd.read_csv(TRAIN_REC / "training_epochs.csv")
display(ep.tail(5))
print(f"epochs: {len(ep)} | best validation loss {ep.val_loss.min():.4f} at epoch {int(ep.loc[ep.val_loss.idxmin(), 'epoch'])} | model files kept: "
      f"{sorted(p.name for p in TRAIN_REC.glob('*.pt')) + sorted(p.name for p in TRAIN_REC.glob('*.ckpt'))}")
ep.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(8, 3), title="loss"); plt.show()

## 4. Test on the generated test set (2000 samples)

In [ ]:
t0 = time.time()
cands = sorted((RESULTS / "main_train_output").glob("**/trained_models/VIT_model.pt"))
assert cands, "the team's trained_models/VIT_model.pt was not found: did training finish?"
TRAIN_RUN_DIR = cands[-1].parent.parent
print("training run dir:", TRAIN_RUN_DIR)
WEIGHTS = TRAIN_REC / ("best_val_model.pt" if EVAL_WEIGHTS == "best_val" else "last_model.pt")
assert WEIGHTS.exists(), WEIGHTS
eval_cmd = [sys.executable, REPO_DIR / "kaggle_185" / "eval_185.py", TEST_NAME,
    "-root_simu", DATA_ROOT / "simulation" / "fsaverage", "-results_path", RESULTS / "eval_output",
    "-eval_simu_type", "sereega", "-source_space", "fsav_994", "-electrode_montage", "standard_1020", "-orientation", "constrained",
    "-leadfield_mat", LEADFIELD, "-to_load", N_TEST, "-per_valid", 1.0, "-n_times", 500, "-eeg_snr", EEG_SNR, "-scaler", "linear",
    "-train_run_dir", TRAIN_RUN_DIR, "-ckpt_path", f"eeg_vit:{WEIGHTS}", "-mets", "eeg_vit"]
run(eval_cmd, "test_eval.log", env={"ESI185_TEST_RECORDS_DIR": str(TEST_REC)}, cwd=REPO_DIR / "model_training")
STEP_TIMES["test_s"] = time.time() - t0
print(json.dumps(json.load(open(TEST_REC / "test_summary.json"))["eeg_vit"], indent=1)[:1800])

## 5. Figures and summary report

In [ ]:
manifest["timings_s"] = {k: round(v, 1) for k, v in STEP_TIMES.items()} | {"total_so_far_s": round(time.time() - T_START, 1)}
manifest["weights_tested"] = str(WEIGHTS.name)
(RESULTS / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
run([sys.executable, REPO_DIR / "kaggle_185" / "analyze_185.py", "--results", RESULTS], "analysis.log")
from IPython.display import Image, display
for f in ("fig_training_curves.png", "fig_test_metrics.png", "fig_test_predictions.png"):
    if (FIGS / f).exists():
        display(Image(str(FIGS / f)))
print((RESULTS / "SUMMARY.md").read_text()[:3500])

## 6. Zip the results folder (download this file)

In [ ]:
ZIP = WORK / "results_185.zip"
if ZIP.exists():
    ZIP.unlink()
run([sys.executable, REPO_DIR / "kaggle_185" / "package_results_185.py", "--results", RESULTS, "--zip", ZIP], "packaging.log")
manifest["timings_s"]["total_s"] = round(time.time() - T_START, 1)
(RESULTS / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nDONE in {(time.time() - T_START) / 3600:.2f} h -> download {ZIP} ({ZIP.stat().st_size / 2**20:.0f} MB) from the Output tab (right panel -> Output).")